# Prepare the arXiv dataset

This notebook downloads or reuses the Kaggle arXiv metadata, samples papers, and creates reusable text vectors. It does not fit UMAP or IMSTE. Run it when the arXiv cache is missing or when you want to increase its size. A larger request replaces the one saved dataset; a smaller request reuses a prefix without saving a second copy.

## 1. Install dependencies

Run once per environment. Kaggle authentication must already be configured.

In [7]:
#%pip install -e ".[notebook]" sentence-transformers kaggle

## 2. Get the metadata snapshot

The JSONL snapshot is large and stays under the ignored `data/arxiv/` directory.

In [8]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import json
import subprocess

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data" / "arxiv"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def is_complete_jsonl(path):
    """Check the first and final records so interrupted extractions are not reused."""
    try:
        if not path.is_file() or path.stat().st_size == 0:
            return False
        with path.open("rb") as file:
            first_line = file.readline()
            file.seek(max(0, path.stat().st_size - 8 * 1024 * 1024))
            tail_lines = file.read().splitlines()
        json.loads(first_line)
        json.loads(tail_lines[-1])
        return True
    except (OSError, ValueError, json.JSONDecodeError, IndexError):
        return False


DOWNLOAD_METADATA = True  # Download automatically if the local JSONL is missing or incomplete.
FORCE_METADATA_DOWNLOAD = False  # Set True only when you intentionally want a fresh snapshot.
metadata_files = sorted(DATA_DIR.glob("*.json"))
jsonl_path = next((path for path in metadata_files if "metadata" in path.name.lower()), None)
metadata_is_valid = jsonl_path is not None and is_complete_jsonl(jsonl_path)

if DOWNLOAD_METADATA and (FORCE_METADATA_DOWNLOAD or not metadata_is_valid):
    if FORCE_METADATA_DOWNLOAD or jsonl_path is not None:
        print("Removing incomplete local download before fetching a clean copy.")
        for path in DATA_DIR.glob("*.json"):
            path.unlink(missing_ok=True)
        for path in DATA_DIR.glob("*.zip"):
            path.unlink(missing_ok=True)
        for path in DATA_DIR.glob("*.kaggle-partial"):
            path.unlink(missing_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "Cornell-University/arxiv",
         "-p", str(DATA_DIR), "--unzip", "--force"],
        check=True,
    )
    metadata_files = sorted(DATA_DIR.glob("*.json"))
    jsonl_path = next((path for path in metadata_files if "metadata" in path.name.lower()), None)
    metadata_is_valid = jsonl_path is not None and is_complete_jsonl(jsonl_path)

print("Data directory:", DATA_DIR)
print("JSON files:", [path.name for path in metadata_files])
if not metadata_is_valid:
    raise RuntimeError(
        "The arXiv metadata JSONL is missing or incomplete. Check Kaggle's download output, "
        "available disk space, and then rerun this cell."
    )


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Data directory: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/arxiv
JSON files: ['arxiv-metadata-oai-snapshot.json']


## 3. Choose the cache size and prepare vectors

Set `MAX_PAPERS` to the desired number of papers. If that is larger than the cached dataset, the new vectors replace the old cache. If it is smaller, the notebook uses the first rows of the existing deterministic sample in memory and leaves the on-disk cache intact.

In [9]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from arxiv_dataset import prepare_arxiv_dataset

RANDOM_STATE = 42
MAX_PAPERS = 320_000
JSON_CHUNK_SIZE = 50_000
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FORCE_REBUILD = False

papers, X, manifest = prepare_arxiv_dataset(
    DATA_DIR,
    jsonl_path,
    max_papers=MAX_PAPERS,
    random_state=RANDOM_STATE,
    json_chunk_size=JSON_CHUNK_SIZE,
    embedding_model=EMBEDDING_MODEL,
    force_rebuild=FORCE_REBUILD,
)
print(f"Prepared {len(papers):,} papers × {X.shape[1]:,} features")
print(f"Vector matrix in memory: {X.nbytes / 1024**3:.2f} GiB")
print("Dataset cache ID:", manifest["dataset_id"])


Sampling up to 320,000 papers from arxiv-metadata-oai-snapshot.json and embedding with sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Saved one 320,000-paper cache at /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/arxiv/cache (384 features; 491,520,000 vector bytes).
Prepared 320,000 papers × 384 features
Vector matrix in memory: 0.46 GiB
Dataset cache ID: e36e7b2c3d0d04d9448f
